In [ ]:
#simple linear regression 

In [1]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
import logging
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# =========================
# 2. Logging Setup
# =========================
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# =========================
# 3. Utility Functions
# =========================
def evaluate_model(y_true, y_pred):
    """Evaluate model using multiple metrics"""
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }

def print_metrics(name, metrics):
    logging.info(f"{name} Performance:")
    for key, value in metrics.items():
        logging.info(f"{key}: {value:.4f}")

# =========================
# 4. Model Builders
# =========================
def build_linear_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ])

def build_ridge_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge())
    ])

# =========================
# 5. Training Function
# =========================
def train_models(X, y):
    # Reproducibility
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    logging.info("Data split completed")

    # -------- Linear Regression --------
    lr = build_linear_pipeline()
    lr.fit(X_train, y_train)

    pred_lr = lr.predict(X_test)
    lr_metrics = evaluate_model(y_test, pred_lr)
    print_metrics("Linear Regression", lr_metrics)

    # Cross-validation
    lr_cv = cross_val_score(lr, X_train, y_train, cv=5, scoring="r2")
    logging.info(f"Linear Regression CV R2: {lr_cv.mean():.4f}")

    # -------- Ridge Regression (with tuning) --------
    ridge = build_ridge_pipeline()

    param_grid = {
        "model__alpha": [0.01, 0.1, 1, 10, 100]
    }

    grid = GridSearchCV(ridge, param_grid, cv=5, scoring="r2", n_jobs=-1)
    grid.fit(X_train, y_train)

    best_ridge = grid.best_estimator_
    logging.info(f"Best Ridge Alpha: {grid.best_params_}")

    pred_ridge = best_ridge.predict(X_test)
    ridge_metrics = evaluate_model(y_test, pred_ridge)
    print_metrics("Ridge Regression", ridge_metrics)

    # Save best model
    joblib.dump(best_ridge, "best_model.pkl")
    logging.info("Best model saved as best_model.pkl")

    return best_ridge

# =========================
# 6. Main Execution
# =========================
if __name__ == "__main__":
    # Example dataset (replace with your actual data)
    # X = pd.read_csv("features.csv")
    # y = pd.read_csv("target.csv")

    # Dummy data for testing
    from sklearn.datasets import make_regression
    X, y = make_regression(n_samples=1000, n_features=10, noise=0.1, random_state=42)

    model = train_models(X, y)

ModuleNotFoundError: No module named 'numpy'